# Practical PyTorch · Part 20 — Companion Notebook

The transformer capstone: turn embeddings into a semantic search engine that finds results by meaning, then wrap it in Gradio. Inference only.

In [ ]:
!pip install -q sentence-transformers gradio

## 1. Embed the corpus once

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

docs = [
    "How to reset your account password",
    "Updating your billing and payment method",
    "Steps to recover a locked account",
    "Exporting your data as a CSV file",
    "Turning on two-factor authentication",
]
doc_embeddings = model.encode(docs, convert_to_tensor=True)
print(doc_embeddings.shape)   # [5, 384]

## 2. Search by meaning

In [ ]:
def search(query):
    if not query.strip():
        return {}
    q = model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q, doc_embeddings)[0]
    top = scores.topk(3)
    return {docs[i]: float(scores[i]) for i in top.indices}

search("I forgot my login")

## 3. Put a search box on it

In [ ]:
import gradio as gr

gr.Interface(
    fn=search,
    inputs=gr.Textbox(label="Search", placeholder="Describe what you're looking for…"),
    outputs=gr.Label(num_top_classes=3, label="Best matches"),
    title="Semantic search",
    description="Finds results by meaning, not keywords.",
).launch(share=True)

**Next: Part 21 — The Hub and the Heavyweights.**